# Phase 6b · step 4 — the locked pass

Step 4 of `preregistration/ANALYSIS_PLAN.md` §10. It runs the same label-free pass as
`07_internal_pass.ipynb` — the same script, the same constants — over the three
**locked** sets, once:

| Job | Images | Role |
|---|---|---|
| `messidor2` | ~1,744 | external — H1′, H2, H3 |
| `aptos` | 3,662 | external — H1′, H2, H3 |
| `eyepacs_test` | 17,615 | in-domain — H1 (custom split, D8) |

The externals run first: they carry the thesis.

## What it cannot do

**It reads no label.** `predict.py` drops every label column the moment a manifest is
read, and writes none. **It cannot run before step 3**: section 2 refuses unless the
fitted parameters are committed, unmodified, and their digest is recorded in
`PREREGISTRATION.md`. **It cannot run on the wrong models**: section 6 checks every
checkpoint's SHA-256 against the ones the parameters were fitted on.

## Inputs

`verify-dr-cache-512` · `verify-dr-manifests` · `verify-dr-phase6` (the six runs) ·
`verify-dr-stage-c` (C2's segmenter). From a second session on, also
`verify-dr-locked` — this notebook's own saved output — so finished shards carry over.

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** — both GPUs are used |
| Persistence | Files only |
| Internet | On |

About **2–4 GPU-hours**: roughly twice the internal pass, because the 19 faithfulness
controls dominate and there are twice as many images. Resumable by shard across sessions.

## 1 · Clone the repo

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
sys.path.insert(0, str(REPO_DIR / "src"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Drop verify_dr modules left over from an earlier checkout in this kernel.
# Python caches modules by NAME, not by file, so re-cloning mid-session does
# nothing for an already-imported package: a later cell importing a function
# added upstream still fails with ImportError against the new files on disk.
for _stale in [m for m in list(sys.modules)
               if m == "verify_dr" or m.startswith("verify_dr.")]:
    del sys.modules[_stale]

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

## 2 · Step 3 happened: the fitted parameters are committed

In [ ]:
# Plan s10, step 3 must come first: the fitted parameters committed and their digest
# recorded in PREREGISTRATION.md. This refuses before a single locked image is read.
from verify_dr.triage.params import step3_record

PARAMS = step3_record(REPO_DIR)
print("step 3 verified: preregistration/fitted_params.json is committed, unmodified,")
print("and its digest is recorded in PREREGISTRATION.md")
print("  digest  ", PARAMS["digest"])
print("  analyses", "registered (primary) + amended (D12, D13)" if "amended" in PARAMS
      else "registered only")

## 3 · GPU check

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Accelerator to 'GPU T4 x2', then re-run from the top.")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory / 2**30:.1f} GB')

## 4 · Helpers

In [ ]:
from pathlib import Path
from collections import Counter
import json, shlex, subprocess, sys, time, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    """Run a training job, streaming its output live.

    The earlier notebooks use capture_output=True, which is fine for a two-minute
    manifest build and useless here: you would see nothing at all until a
    three-hour job exited. Streaming means a per-epoch line appears as it happens,
    so a run that is going wrong can be stopped in epoch 1 rather than hour 3.
    """
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        if code == 2:
            print("\nExit 2 is an argument error. Usually the cloned scripts are stale:")
            print("re-run the clone cell at the top, then run from there.")
        raise RuntimeError(f"training failed with exit {code}")

def cache_roots():
    """Every directory that looks like a build_cache.py output root.

    build_cache.py writes <root>/<dataset>/cache_report.json, so a report file
    identifies its root two levels up. /kaggle/input is searched first: a stale
    copy in /kaggle/working must never silently win over the dataset you attached.
    """
    found = []
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for root, _ in Counter(r.parent.parent for r in base.rglob("cache_report.json")).most_common():
            found.append(root)
    return found

def resolve_datasets(roots):
    """dataset -> the root holding the best copy of it.

    The cache is legitimately split across published datasets: the full build
    plus a later top-up. IDRiD's lesion masks ship as their own dataset
    ('verify-dr-idrid-masks'), so a single root shows idrid with zero mask
    channels even when the masks are attached. When a dataset appears in more
    than one root, the copy with more mask channels wins.
    """
    best = {}
    for root in roots:
        for d in sorted(x for x in root.iterdir() if x.is_dir()):
            if not (d / "cache_report.json").exists():
                continue
            masks = d / "masks"
            score = len(list(masks.iterdir())) if masks.is_dir() else 0
            if d.name not in best or score > best[d.name][1]:
                best[d.name] = (root, score)
    return {name: root for name, (root, _) in best.items()}

def extract_cache(dest=WORK / "cache512"):
    """Extract a cache published as a zip. Idempotent within a session.

    This costs GPU-session minutes, which come out of the 30 h/week quota. If you
    hit it every run, re-upload the cache to Kaggle as a *dataset* rather than as
    notebook output -- an uploaded zip is unpacked by Kaggle once, server-side.
    """
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
        except (zipfile.BadZipFile, OSError):
            continue
        if not any(n.endswith("cache_report.json") for n in names):
            continue
        marker = dest / ".extracted_from"
        if marker.exists() and marker.read_text().strip() == z.name:
            print(f"already extracted from {z.name}")
            return dest
        print(f"extracting {z.name} ({z.stat().st_size / 2**30:.1f} GB) -> {dest}")
        dest.mkdir(parents=True, exist_ok=True)
        started = time.time()
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        marker.write_text(z.name)
        print(f"extracted in {(time.time() - started) / 60:.1f} min")
        return dest
    return None

def find_manifest_dir():
    """Phase 2's output: the directory holding dataset_plan.json and the variants."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        hits = sorted(base.rglob("dataset_plan.json"))
        if hits:
            return hits[0].parent
    return None

In [ ]:
def run_pass(cmd):
    """Like run(), but exit 3 means 'stopped at --max-minutes; re-run to resume'."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code not in (0, 3):
        raise RuntimeError(f"predict.py failed with exit {code}")
    return code

## 5 · Cache and manifests

In [ ]:
ROOTS = cache_roots()
if not ROOTS and extract_cache():
    ROOTS = cache_roots()
MANIFEST_DIR = find_manifest_dir()
if not ROOTS:
    raise RuntimeError("No cache found. Attach verify-dr-cache-512 under Add Data.")
if MANIFEST_DIR is None:
    raise RuntimeError("No manifests found. Attach verify-dr-manifests.")
CACHE_FLAGS = ' '.join(q(r) for r in ROOTS)

FULL = MANIFEST_DIR / 'eyepacs_full.csv'
APTOS = MANIFEST_DIR / 'aptos_external.csv'
MESSIDOR2 = MANIFEST_DIR / 'messidor2_external.csv'
for m in (FULL, APTOS, MESSIDOR2):
    if not m.exists():
        raise RuntimeError(f"{m.name} missing from {MANIFEST_DIR}")
print('cache roots :', *ROOTS, sep='\n   ')
print('manifests   :', MANIFEST_DIR)

## 6 · The checkpoints — the ones the parameters were fitted on

In [ ]:
# The six Phase 6a runs and C2's segmenter, found by name wherever they are mounted.
RUNS = [f'H1_{v}_s{s}' for v in ('eyepacs_full', 'eyepacs_ddr_full') for s in (42, 43, 44)]
found = {}
for p in sorted(INPUT.rglob('H1_*/best.pt')):
    found.setdefault(p.parent.name, p)
missing = [r for r in RUNS if r not in found]
if missing:
    raise RuntimeError(f'missing best.pt for {missing}. Attach the LATEST version of '
                       'verify-dr-phase6 -- an older one holds only the first three runs.')
M1 = [found[r] for r in RUNS]
M1_FULL = [found[r] for r in RUNS if '_ddr_' not in r]
M1_DDR = [found[r] for r in RUNS if '_ddr_' in r]

c2 = sorted(INPUT.rglob('C2_lesions/best.pt'))
if not c2:
    raise RuntimeError("C2's segmenter not found. Attach verify-dr-stage-c.")
M2 = c2[0]

for r in RUNS:
    print(f'  {r:<26} {found[r].stat().st_size / 2**20:5.1f} MB  {found[r].parent}')
print(f'  {"C2_lesions (M2)":<26} {M2.stat().st_size / 2**20:5.1f} MB  {M2.parent}')

In [ ]:
# Every checkpoint must be the one fit_params.py fitted against. A different version
# of verify-dr-phase6 or verify-dr-stage-c would produce a pass analyse.py refuses at
# the unblinding -- after the GPU hours were spent. Checked here, before any.
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

wrong = []
for run in RUNS:
    if sha256(found[run]) != PARAMS['models'][run]['checkpoint_sha256']:
        wrong.append(run)
if sha256(M2) != PARAMS['evidence_checkpoint_sha256']:
    wrong.append('C2_lesions (M2)')
if wrong:
    raise RuntimeError(f'checkpoints differ from the fitted ones: {wrong}. Attach the '
                       'versions of verify-dr-phase6 and verify-dr-stage-c used by the '
                       'internal pass.')
print('all seven checkpoints match the fitted parameters')

## 7 · Budget

In [ ]:
import time
# Kaggle's weekly GPU counter, honestly. The pass stops cleanly between shards
# rather than being killed mid-shard; a stopped job resumes where it left off.
GPU_HOURS_LEFT = 25.0
SESSION_LIMIT_H = 11.0
SAFETY_MARGIN_H = 0.3
SESSION_START = time.time()

def minutes_left():
    usable_h = min(GPU_HOURS_LEFT, SESSION_LIMIT_H) - SAFETY_MARGIN_H
    return usable_h * 60 - (time.time() - SESSION_START) / 60

print(f'usable this session: {minutes_left():.0f} minutes')

## 8 · Carry forward a stopped run

In [ ]:
# /kaggle/working does not survive a session. A job that stopped part-way resumes from
# its finished shards -- but only if they come back in, from verify-dr-locked.
import shutil

OUT = WORK / 'locked'
OUT.mkdir(parents=True, exist_ok=True)
for run_json in sorted(INPUT.rglob('locked/*/run.json')):
    job = run_json.parent
    dest = OUT / job.name
    if dest.exists():
        continue
    shutil.copytree(job, dest)
    state = 'complete' if (dest / 'images.csv').exists() else 'partial -- will resume'
    print(f'carried forward {job.name}: {state}')
print('output:', OUT)

## 9 · The locked pass

`--locked` is passed here and nowhere else in the project. Each job resumes from its
finished shards and stops cleanly between shards when the budget runs low; re-running
this cell is always safe.

In [ ]:
EVIDENCE = ['--evidence-checkpoint', q(M2)]
JOBS = [
    # The externals first: they decide H1', the thesis.
    ('messidor2', MESSIDOR2, 'test'),
    ('aptos', APTOS, 'test'),
    ('eyepacs_test', FULL, 'test'),
]

stopped = False
for name, manifest, split in JOBS:
    if (OUT / name / 'images.csv').exists():
        print(f'{name}: complete already, skipping')
        continue
    budget = minutes_left() - 10
    if budget < 5:
        print(f'\n{name}: not started -- {minutes_left():.0f} minutes left this session.')
        stopped = True
        break
    print('\n' + '=' * 72 + f'\n{name}   ({budget:.0f} minutes of budget)\n' + '=' * 72)
    code = run_pass(' '.join([
        f"python {q(REPO_DIR / 'scripts/predict.py')}",
        f'--manifest {q(manifest)} --split {split} --locked',
        '--grading-checkpoints ' + ' '.join(q(c) for c in M1),
        *EVIDENCE,
        f'--cache-root {CACHE_FLAGS}',
        f'--out-dir {q(OUT / name)}',
        f'--max-minutes {budget:.0f} --workers 4',
    ]))
    if code == 3:
        print(f'\n{name} stopped part-way. Quick Save, publish, and re-run next session.')
        stopped = True
        break

done = [n for n, *_ in JOBS if (OUT / n / 'images.csv').exists()]
print(f'\ncomplete: {done}')
if not stopped and len(done) == len(JOBS):
    print('All three locked jobs complete. The locked pass is DONE.')

## 10 · What the pass holds — without a label

Descriptive and label-free: how many images, what M3 found, how the faithfulness test
went. Nothing here can be scored, because nothing here knows the truth.

In [ ]:
import json as _json
import pandas as pd

for name, *_ in JOBS:
    path = OUT / name / 'images.csv'
    if not path.exists():
        print(f'{name}: not complete\n')
        continue
    images = pd.read_csv(path)
    run = _json.loads((OUT / name / 'run.json').read_text())
    lesions = images['faith_status'] != 'none'
    print(f"{name}: {len(images)} images; locked={run['locked']}; "
          f"labels_written={run['labels_written']}")
    print(f"  evidence grade   {images['evidence_grade'].value_counts().sort_index().to_dict()}")
    print(f"  rules fired      {images['rule'].value_counts().to_dict()}")
    print(f"  with lesions     {int(lesions.sum())} ({lesions.mean():.0%})")
    print(f"  faithfulness     {images['faith_status'].value_counts().to_dict()}")
    if lesions.any():
        ok = images.loc[lesions, 'controls_ok']
        print(f'  controls placed  median {ok.median():.0f} of 19, min {ok.min()}')
    print()

---
## 11 · Save

1. **Save Version → Quick Save.** Never *Save & Run All*.
2. Publish `/kaggle/working/locked` as **`verify-dr-locked`**.
3. If section 9 stopped part-way, attach `verify-dr-locked` next session and run again.

Paste back the outputs of **sections 9 and 10**. Then run `10_unblinding.ipynb`, the one
label join, on CPU.